#Transcrição OpenAI

https://www.kaggle.com/code/luisguimaraes/assistente-whisper-chatgpt-e-elevenlabs-api

In [ ]:
language = 'pt'

---Instalando o ffmpeg para conversão de áudio---
%pip install ffmpeg-python

---Instalando a biblioteca da openai para a resposta---
%pip install openai

---Instalando a biblioteca da ElevenLABS para sintetização da voz do texto de resposta---
%pip install elevenlabs

---Instalando o Whisper para reconhecimento de voz---
%pip install git+https://github.com/openai/whisper.git

In [ ]:
pip install ffmpeg-python

In [ ]:
pip install openai-whisper

In [ ]:
%pip install openai

In [ ]:
#Importando o Whisper
import whisper

#Iniciando o reconhecimento de voz
print("Entendendo.....")

#Selecionando o modelo de reconhecimento de voz
model = whisper.load_model("medium")

#Transcrevendo o áudio gravado
resultado = model.transcribe("/audio/minha_gravacao.wav", fp16 = False, language = language)
transcricao = resultado["text"]

#Imprimindo a transcrição para validação
print("\nVocê disse: ")
print(transcricao)

#Bibliotecas

In [ ]:
import whisper #transcrição
import pandas as pd
import nltk
nltk.download('punkt')
nltk.download('stopwords')
from nltk.corpus import stopwords
from nltk.tokenize import word_tokenize
import re
import string
from nltk.stem import WordNetLemmatizer
nltk.download('wordnet')

ModuleNotFoundError: No module named 'whisper'

#Transcrevendo audio

O WER é calculado como a soma das substituições, inserções e deleções dividida pelo número total de palavras no texto de referência.

Quanto menor o WER, mais precisa é a transcrição.

Um WER de 0% significa que a transcrição é idêntica ao texto de referência.

In [ ]:
def wer(reference, hypothesis):
    # Implementação do algoritmo de Levenshtein para calcular o WER
    # Converte a referência em uma lista de palavras
    ref_words = reference.split()

    # Verifica se a hipótese é uma string ou um dicionário
    if isinstance(hypothesis, str):
        # Se for uma string, converte em uma lista de palavras
        hyp_words = hypothesis.split()
    elif isinstance(hypothesis, dict):
        # Se for um dicionário, verifica se possui a chave 'Text'
        if 'Text' in hypothesis:
            # Converte o texto da hipótese em uma lista de palavras
            hyp_words = hypothesis['Text'].split()
        else:
            raise ValueError("O dicionário de hipótese não contém a chave 'Text'")
    else:
        raise ValueError("O tipo de hipótese fornecido não é suportado")

    # Obtém o comprimento das transcrições de referência e hipótese
    ref_len = len(ref_words)
    hyp_len = len(hyp_words)

    # Inicializa a matriz de distâncias
    distance = [[0] * (hyp_len + 1) for _ in range(ref_len + 1)]

    # Preenche a primeira linha e a primeira coluna
    for i in range(ref_len + 1):
        distance[i][0] = i
    for j in range(hyp_len + 1):
        distance[0][j] = j

    # Calcula a distância de edição
    for i in range(1, ref_len + 1):
        for j in range(1, hyp_len + 1):
            if ref_words[i - 1] == hyp_words[j - 1]:
                cost = 0
            else:
                cost = 1
            distance[i][j] = min(distance[i - 1][j] + 1,  # Deleção
                                 distance[i][j - 1] + 1,  # Inserção
                                 distance[i - 1][j - 1] + cost)  # Substituição

    # Calcula o número total de operações de edição
    edits = distance[ref_len][hyp_len]

    # Calcula S (substituições), D (exclusões), I (inserções)
    S = distance[ref_len][hyp_len]
    D = ref_len - S
    I = hyp_len - S

    # Calcula o WER
    wer = float(S + D + I) / ref_len

    return wer, S, D, I

WER: 2.00
Substituições: 0, Exclusões: 211, Inserções: 211


In [ ]:
from nltk.metrics import edit_distance

def calcular_precisao(transcricao_referencia, transcricao_sistema):
    ref_words = transcricao_referencia.split()
    hyp_words = transcricao_sistema.split()

    ref_len = len(ref_words)
    hyp_len = len(hyp_words)

    distancia = edit_distance(transcricao_referencia.split(), transcricao_sistema.split())
    total_palavras = len(transcricao_referencia.split())
    precisao = (total_palavras - distancia) / total_palavras * 100
    return precisao

## Transcrição Small
- precisão de 68,75%
- Caluclo WER: 1,78 - Quanto mais perto de 0 melhor

In [ ]:
modelo = whisper.load_model("small")
transcricao_small = modelo.transcribe("2962625.wav")

print(transcricao_small['text'])

/usr/local/lib/python3.10/dist-packages/whisper/transcribe.py:126: UserWarning: FP16 is not supported on CPU; using FP32 instead
  warnings.warn("FP16 is not supported on CPU; using FP32 instead")


 Bom dia, meu nome é Beatriz, falo da Toto, falo do senhor Gabriel Lopes. Não, mas eu posso te ensinar a ligação para ele, sabe o instante. Obrigada. É de onde que é mesmo você falou? Da Toto. Toto, está bom. Obrigada. Técnica. Bom dia, falo com o senhor Gabriel. Isso. O senhor Gabriel me chama Beatriz, falo da Toto, tudo bem? Tudo bem, Beatriz, posso te ajudar? Bom, o senhor Gabriel, o senhor seria ainda um dos seus responsáveis pelo sistema na técnica engenharia? Não, não está mais atualmente comigo, sobre o que seria, talvez eu possa ajudar? Eu preciso fazer um acompanhamento rápido, porém importante, o senhor Gabriel, só para saber como está a experiência, a satisfação de vocês, o senhor seria do faturamento e utilizar o sistema ainda? Ah, sim, então a gente está deixando o sistema, a gente está ensinando um contrato. Ah, entendi, então vocês não utilizam mais? Não, desde fevereiro a gente parou de utilizar, então a gente está utilizando mesmo só para consulta dos quidos, nós já es

In [ ]:
referencia = 'Bom dia, meu nome é Beatriz, eu falo da TOTUS, falo com o senhor Gabriel Lopes? Não, mas eu posso transferir a ligação pra ele, só um instante. Tá bom, obrigada. É de onde que é mesmo, que você falou? da TOTUS. TOTUS tá bom. Obrigada. Nada. Técnica?. Bom dia, falo com o senhor Gabriel? Isso. Senhor Gabriel, me chamo Beatriz, falo da TOTUS tudo bem? Tudo bem Beatriz, como posso te ajudar? Bom senhor gabriel o senhor seria ainda um dos responsáveis pelo sistema na técnica e engenharia? Não, não ta mais atualmente comigo, sobre o que seria, talvez eu possa ajudar? Eu preciso fazer um acompanhamento rápido porém importante senhor Gabriel, só pra saber como está a experiência e satisfação de vocês, o senhor seria do faturamento e utiliza o sistema ainda? Ah, sim, então a gente ta deixando o sistema, a gente ta encerrando o contrato. Ah, entendi, então vocês não utilizam mais? Não, não, desde fevereiro a gente parou de utilizar, atualmente a gente ta utilizando mesmo só para consulta dos arquivos que já estão feitos né, mas a gente encerrou o contrato com a TOTUS. Ah, então tá bom, senhor Gabriel. Eu agradeço e tenha um ótimo dia, tchau tchau. Até mais tchau.'

In [ ]:
transcricao = transcricao_small['text']

In [ ]:
wer_score, substitutions, deletions, insertions = wer(reference_transcription, hypothesis_transcription)
print("WER: {:.2f}".format(wer_score))
print("Substituições: {}, Exclusões: {}, Inserções: {}".format(substitutions, deletions, insertions))

WER: 1.73
Substituições: 60, Exclusões: 148, Inserções: 151


In [ ]:
precisao = calcular_precisao(referencia, transcricao)
print("Precisão da transcrição:", precisao, "%")

Precisão da transcrição: 68.75 %


##Transcrição medium
- precisão de 73%
- Caluclo WER: 1,66 - Quanto mais perto de 0 melhor

audio: 1:33

tempo de execução: 5:30

In [ ]:
modelo = whisper.load_model("medium")
transcricao_medium = modelo.transcribe("2962625.wav")

print(transcricao_medium['text'])

100%|█████████████████████████████████████| 1.42G/1.42G [00:39<00:00, 38.5MiB/s]
/usr/local/lib/python3.10/dist-packages/whisper/transcribe.py:115: UserWarning: FP16 is not supported on CPU; using FP32 instead
  warnings.warn("FP16 is not supported on CPU; using FP32 instead")


 Bom dia, meu nome é Beatriz, eu falo da TOTUS, fala o senhor Gabriel Lopes? Não, mas eu posso estar precisando de uma ligação pra ele, só um instante. Tá bom, obrigada. É de onde que é mesmo, você falou? Da TOTUS. TOTUS, tá bom. Obrigada. Nada. Técnica? Bom dia, falo com o senhor Gabriel. Isso. O senhor Gabriel, me chamo Beatriz, falo da TOTUS, tudo bem? Tudo bem, Beatriz, posso te ajudar? Bom. O senhor Gabriel, o senhor seria ainda um dos responsáveis pelo sistema na técnica e engenharia? Não, não tá mais atualmente comigo, sobre o que seria, talvez eu possa ajudar? Eu preciso fazer um acompanhamento rápido, porém importante, o senhor Gabriel, só pra saber como está a experiência e a satisfação de vocês, o senhor seria do faturamento e utiliza o sistema ainda? Ah, sim, então, a gente tá deixando o sistema, a gente tá ensaiando o contrato. Ah, entendi, então vocês não utilizam mais? Não, não, desde fevereiro a gente parou de utilizar, atualmente a gente tá utilizando mesmo só pra cons

In [ ]:
referencia = 'Bom dia, meu nome é Beatriz, eu falo da TOTUS, falo com o senhor Gabriel Lopes? Não, mas eu posso transferir a ligação pra ele, só um instante. Tá bom, obrigada. É de onde que é mesmo, que você falou? da TOTUS. TOTUS tá bom. Obrigada. Nada. Técnica?. Bom dia, falo com o senhor Gabriel? Isso. Senhor Gabriel, me chamo Beatriz, falo da TOTUS tudo bem? Tudo bem Beatriz, como posso te ajudar? Bom senhor gabriel o senhor seria ainda um dos responsáveis pelo sistema na técnica e engenharia? Não, não ta mais atualmente comigo, sobre o que seria, talvez eu possa ajudar? Eu preciso fazer um acompanhamento rápido porém importante senhor Gabriel, só pra saber como está a experiência e satisfação de vocês, o senhor seria do faturamento e utiliza o sistema ainda? Ah, sim, então a gente ta deixando o sistema, a gente ta encerrando o contrato. Ah, entendi, então vocês não utilizam mais? Não, não, desde fevereiro a gente parou de utilizar, atualmente a gente ta utilizando mesmo só para consulta dos arquivos que já estão feitos né, mas a gente encerrou o contrato com a TOTUS. Ah, então tá bom, senhor Gabriel. Eu agradeço e tenha um ótimo dia, tchau tchau. Até mais tchau.'

In [ ]:
transcricao = transcricao_medium['text']

In [ ]:
precisao = calcular_precisao(referencia, transcricao)
print("Precisão da transcrição:", precisao, "%")

Precisão da transcrição: 73.07692307692307 %


In [ ]:
wer_score, substitutions, deletions, insertions = wer(referencia, transcricao)
print("WER: {:.2f}".format(wer_score))
print("Substituições: {}, Exclusões: {}, Inserções: {}".format(substitutions, deletions, insertions))

NameError: name 'wer' is not defined

##Transcrição Large
- precisão:
- Caluclo WER:  - Quanto mais perto de 0 melhor

In [ ]:
modelo = whisper.load_model("large")
transcricao_large = modelo.transcribe("2962625.wav")

print(transcricao_large['text'])

NameError: name 'whisper' is not defined

In [ ]:
referencia = 'Bom dia, meu nome é Beatriz, eu falo da TOTUS, falo com o senhor Gabriel Lopes? Não, mas eu posso transferir a ligação pra ele, só um instante. Tá bom, obrigada. É de onde que é mesmo, que você falou? da TOTUS. TOTUS tá bom. Obrigada. Nada. Técnica?. Bom dia, falo com o senhor Gabriel? Isso. Senhor Gabriel, me chamo Beatriz, falo da TOTUS tudo bem? Tudo bem Beatriz, como posso te ajudar? Bom senhor gabriel o senhor seria ainda um dos responsáveis pelo sistema na técnica e engenharia? Não, não ta mais atualmente comigo, sobre o que seria, talvez eu possa ajudar? Eu preciso fazer um acompanhamento rápido porém importante senhor Gabriel, só pra saber como está a experiência e satisfação de vocês, o senhor seria do faturamento e utiliza o sistema ainda? Ah, sim, então a gente ta deixando o sistema, a gente ta encerrando o contrato. Ah, entendi, então vocês não utilizam mais? Não, não, desde fevereiro a gente parou de utilizar, atualmente a gente ta utilizando mesmo só para consulta dos arquivos que já estão feitos né, mas a gente encerrou o contrato com a TOTUS. Ah, então tá bom, senhor Gabriel. Eu agradeço e tenha um ótimo dia, tchau tchau. Até mais tchau.'

In [ ]:
transcricao = transcricao_large['text']

In [ ]:
precisao = calcular_precisao(referencia, transcricao)
print("Precisão da transcrição:", precisao, "%")

In [ ]:
wer_score, substitutions, deletions, insertions = wer(referencia, transcricao)
print("WER: {:.2f}".format(wer_score))
print("Substituições: {}, Exclusões: {}, Inserções: {}".format(substitutions, deletions, insertions))

# Pré processamento do texto transcrito

In [ ]:
transcricao = transcricao.lower()
print(transcricao)

 bom dia, meu nome é beatriz, eu falo da totus, fala o senhor gabriel lopes? não, mas eu posso estar precisando de uma ligação pra ele, só um instante. tá bom, obrigada. é de onde que é mesmo, você falou? da totus. totus, tá bom. obrigada. nada. técnica? bom dia, falo com o senhor gabriel. isso. o senhor gabriel, me chamo beatriz, falo da totus, tudo bem? tudo bem, beatriz, posso te ajudar? bom. o senhor gabriel, o senhor seria ainda um dos responsáveis pelo sistema na técnica e engenharia? não, não tá mais atualmente comigo, sobre o que seria, talvez eu possa ajudar? eu preciso fazer um acompanhamento rápido, porém importante, o senhor gabriel, só pra saber como está a experiência e a satisfação de vocês, o senhor seria do faturamento e utiliza o sistema ainda? ah, sim, então, a gente tá deixando o sistema, a gente tá ensaiando o contrato. ah, entendi, então vocês não utilizam mais? não, não, desde fevereiro a gente parou de utilizar, atualmente a gente tá utilizando mesmo só pra cons

In [ ]:
transcricao.split(" ")

In [ ]:
stopwords = set(stopwords.words('portuguese'))
len(stopwords)

207

In [ ]:
def limpar_texto(texto):
    texto = texto.lower()
    texto = texto.translate(str.maketrans('', '', string.punctuation))
    texto = re.sub(r'\d+', '', texto)
    texto = texto.strip()

    # Tokenização
    palavras = texto.split()
    palavras = [palavra for palavra in palavras if palavra not in stopwords]

    # Lematização
    lemmatizer = WordNetLemmatizer()
    palavras = [lemmatizer.lemmatize(palavra) for palavra in palavras]

    texto_limpo = ' '.join(palavras)

    return texto_limpo

In [ ]:
limpar_texto(transcricao)

'bom dia nome beatriz falo totus fala senhor gabriel lope posso precisando ligação pra instante tá bom obrigada onde falou totus totus tá bom obrigada nada técnica bom dia falo senhor gabriel senhor gabriel chamo beatriz falo totus tudo bem tudo bem beatriz posso ajudar bom senhor gabriel senhor ainda responsáveis sistema técnica engenharia tá atualmente comigo sobre talvez possa ajudar preciso fazer acompanhamento rápido porém importante senhor gabriel pra saber experiência satisfação senhor faturamento utiliza sistema ainda ah sim então gente tá deixando sistema gente tá ensaiando contrato ah entendi então utilizam desde fevereiro gente parou utilizar atualmente gente tá utilizando pra consulta ah então tá bom senhor gabriel agradeço tchau tchau tchau tchau'

In [ ]:
texto_limpo = limpar_texto(transcricao)